# 07 · Gerçek, yazar-ayrık HTR karşılaştırması

Aynı test yazarlarında Keras CTC, TrOCR ve güncel PaddleOCR Türkçe modelini CER/WER ile ölçer. Sonuçlar model adıyla birlikte JSON kaydedilir. Bu defter eğitim yapmaz ve test verisini modele göstermez.

In [ ]:
!pip install -q "tensorflow>=2.16,<2.22" "transformers<5" jiwer pillow paddleocr
from google.colab import drive
drive.mount('/content/drive')
!rm -rf /content/gercek_satir && mkdir -p /content/gercek_satir
!unzip -qo /content/drive/MyDrive/sinav_okur/gercek_satir_verisi.zip -d /content/gercek_satir

In [ ]:
import json, time
from pathlib import Path
import numpy as np
from PIL import Image
from jiwer import cer, wer

ROOT=Path('/content/gercek_satir'); LABELS=next(ROOT.rglob('labels.jsonl')); BASE=LABELS.parent
records=[json.loads(x) for x in LABELS.read_text(encoding='utf-8').splitlines() if x.strip()]
for r in records: r['image']=str(Path(r['image']) if Path(r['image']).is_absolute() else BASE/r['image'])
MODEL_DIR=Path('/content/drive/MyDrive/sinav_okur/modeller/gercek_satir_ctc')
splits=json.loads((MODEL_DIR/'writer_splits.json').read_text(encoding='utf-8'))
test=[r for r in records if r['writer_id'] in set(splits['test'])]
refs=[r['text'] for r in test]
assert test and set(splits['train']).isdisjoint(splits['test'])
print(len(test),'test satırı',len(set(r['writer_id'] for r in test)),'görülmemiş yazar')

In [ ]:
import cv2, tensorflow as tf
keras_model=tf.keras.models.load_model(MODEL_DIR/'turkce_gercek_satir_ctc.keras',compile=False)
chars=json.loads((MODEL_DIR/'ctc_etiketler.json').read_text(encoding='utf-8')); blank=len(chars)
def prep(path,H=64,W=1024):
    g=cv2.imread(path,cv2.IMREAD_GRAYSCALE); h,w=g.shape; s=min(H/h,W/w); nw,nh=max(1,round(w*s)),max(1,round(h*s))
    z=cv2.resize(g,(nw,nh)); canvas=np.full((H,W),255,np.uint8); canvas[(H-nh)//2:(H-nh)//2+nh,:nw]=z
    return (1-canvas.astype('float32')/255)[...,None]
def decode(prob):
    row=prob.argmax(-1); prev=None; out=[]
    for i in row:
        i=int(i)
        if i!=blank and i!=prev: out.append(chars[i])
        prev=i
    return ''.join(out).strip()
def run_keras():
    x=np.stack([prep(r['image']) for r in test]); return [decode(p) for p in keras_model.predict(x,batch_size=8,verbose=0)]

In [ ]:
import torch
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
TROCR_MODEL='microsoft/trocr-base-handwritten' # Türkçe fine-tune varsa Drive yoluyla değiştirin
processor=TrOCRProcessor.from_pretrained(TROCR_MODEL)
trocr=VisionEncoderDecoderModel.from_pretrained(TROCR_MODEL).to('cuda' if torch.cuda.is_available() else 'cpu').eval()
def run_trocr():
    out=[]
    for r in test:
        px=processor(images=Image.open(r['image']).convert('RGB'),return_tensors='pt').pixel_values.to(trocr.device)
        with torch.no_grad(): seq=trocr.generate(px,max_new_tokens=128)
        out.append(processor.batch_decode(seq,skip_special_tokens=True)[0].strip())
    return out

In [ ]:
from paddleocr import PaddleOCR
paddle=PaddleOCR(lang='tr',use_doc_orientation_classify=False,use_doc_unwarping=False,use_textline_orientation=False)
def run_paddle():
    out=[]
    for r in test:
        sonuc=paddle.predict(r['image'])
        metin=[]
        for parca in sonuc:
            veri=parca.json() if callable(getattr(parca,'json',None)) else getattr(parca,'json',{})
            veri=veri.get('res',veri) if isinstance(veri,dict) else {}
            metin.extend(veri.get('rec_texts',[]))
        out.append(' '.join(metin).strip())
    return out

In [ ]:
runners={'keras_ctc_real':run_keras,'trocr_base_iam':run_trocr,'paddleocr_tr':run_paddle}
rapor={}
for ad,fn in runners.items():
    bas=time.time(); hyp=fn(); rapor[ad]={'cer':float(cer(refs,hyp)),'wer':float(wer(refs,hyp)),'seconds':time.time()-bas,'lines':len(refs),'writer_disjoint':True}
    print(ad,rapor[ad])
kazanan=min(rapor,key=lambda x:rapor[x]['cer']); rapor['winner']=kazanan
out=Path('/content/drive/MyDrive/sinav_okur/modeller/gercek_benchmark.json'); out.write_text(json.dumps(rapor,ensure_ascii=False,indent=2),encoding='utf-8')
print('Kazanan:',kazanan,'→',out)